# Pattern 1: ReAct Tool-Using Agent (LangChain Agents + Ollama)

This notebook demonstrates a classic ReAct-style agent that reasons and uses tools.
It runs fully local with Ollama model `qwen3.5:2b`.

In [ ]:
from datetime import datetime
from typing import Any

from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_ollama import ChatOllama

MODEL_NAME = "qwen3.5:2b"
llm = ChatOllama(model=MODEL_NAME, temperature=0)

In [ ]:
@tool
def current_time(_: str = "") -> str:
    """Return the current local timestamp."""
    return datetime.now().isoformat(timespec="seconds")

@tool
def safe_calculator(expression: str) -> str:
    """Evaluate a simple arithmetic expression, e.g. '(12 * 9) + 7'."""
    allowed = set("0123456789+-*/(). %")
    if any(ch not in allowed for ch in expression):
        return "Only arithmetic characters are allowed."
    try:
        value = eval(expression, {"__builtins__": {}}, {})
        return str(value)
    except Exception as exc:
        return f"Calculation error: {exc}"

tools = [current_time, safe_calculator]

In [ ]:
agent = create_agent(model=llm, tools=tools)

prompt = (
    "What is (27 * 14) + 9? Then tell me the current local timestamp. "
    "Show both answers clearly."
)

result = agent.invoke({"messages": [{"role": "user", "content": prompt}]})
result["messages"][-1].content

## Try your own prompt
Change the user prompt above to force the agent to pick different tools, or ask for a multi-step task.